In [4]:
def compute_temporal_metrics(original_values, wet_threshold=0.1):

    #values = np.asarray(values).flatten()
    #print(values)
    # original_values = np.fromstring(original_values.strip('[]'), sep=' ')
    original_len = len(original_values)
    
    # Remove leading/trailing values below wet threshold
    wet = original_values > wet_threshold
    nonzero = np.flatnonzero(wet)

    if len(nonzero) == 0:
        return {'total_acc': 0,'peak_position_ratio': np.nan,'d50': np.nan,'gini': 0.0,'time_based_std': np.nan,
                'dry_ratio': np.nan,'wet_dry_transition_rate': np.nan,'max_quartile_index': np.nan}

    trimmed_values = original_values[nonzero[0]:nonzero[-1] + 1]
    n = len(trimmed_values)

    ## Amount before / after peak
    peak_idx = np.argmax(trimmed_values)

    rainfall_before_peak = np.sum(trimmed_values[0:peak_idx])
    rainfall_after_peak = np.sum(trimmed_values[peak_idx+1:])
    rainfall_at_peak = trimmed_values[peak_idx]
    total_rainfall = np.sum(trimmed_values)
    
#     if round((rainfall_before_peak + rainfall_after_peak + rainfall_at_peak),3) != round(total_rainfall,4):
#         print(rainfall_before_peak + rainfall_after_peak + rainfall_at_peak)
#         print(total_rainfall)
#         print("Something's wrong, John")

    parts_sum = rainfall_before_peak + rainfall_after_peak + rainfall_at_peak

    if not np.isclose(parts_sum, total_rainfall, rtol=0, atol=1e-3):
        print(parts_sum)
        print(total_rainfall)
        print("Something's wrong, John")

    
    #if n < original_len - 12:
        #print(original_values)
        #print(trimmed_values)
        #print('------------')
        #plt.plot(trimmed_values)
        #plt.show()
        
    # Wet/dry state
    wet = trimmed_values > wet_threshold

    # --- Wet-dry transition rate ---
    if n > 1:
        transitions = np.sum(wet[1:] != wet[:-1])
        wet_dry_transition_rate = transitions / (n - 1)
    else:
        wet_dry_transition_rate = 0.0

    # --- Basic ---
    total_acc = np.nansum(trimmed_values)
    max_precip = np.nanmax(trimmed_values)
    
    ## Modality
    modality_result = classify_modality(trimmed_values)
    
    # --- Accumulations over certain time periods --- 
    dt = 1.0
    vals = pd.Series(np.asarray(trimmed_values, float) * dt)
    n = len(vals)
    max_acc = {h: vals.rolling(min(int(round(h / dt)), n)).sum().max() for h in (2, 3, 6, 12, 16)}        

    # Peak position ratio
    peak_idx = np.argmax(trimmed_values)
    peak_position_ratio = peak_idx / max(n - 1, 1)

    # --- D50 ---
    if total_acc > 0:
        cum = np.cumsum(trimmed_values) / total_acc
        time_pct = np.linspace(0, 100, n)

        idx = np.where(cum >= 0.5)[0]

        if len(idx) > 0:
            i = idx[0]

            if i == 0:
                d50 = time_pct[0]
            else:
                x1, y1 = time_pct[i-1], cum[i-1]
                x2, y2 = time_pct[i], cum[i]

                d50 = (x1 + (0.5 - y1) * (x2 - x1) / (y2 - y1)
                    if (y2 - y1) != 0 else np.nan)
        else:
            d50 = np.nan
    else:
        d50 = np.nan

    # --- Gini coefficient ---
    if n == 0 or np.all(trimmed_values == 0):
        gini = 0.0
    else:
        s = np.sort(trimmed_values)
        idx = np.arange(1, n + 1)
        gini = (2 * np.sum(idx * s) / (n * np.sum(s))) - (n + 1) / n

    # --- Time-based std ---
    if total_acc > 0:
        pos = np.linspace(0, 1, n)
        t_cg = np.sum(pos * trimmed_values) / total_acc
        time_based_std = np.sqrt(np.sum(((pos - t_cg) ** 2) * trimmed_values) / total_acc)
    else:
        time_based_std = np.nan

    # --- Event dry ratio ---
    rounded = np.round(trimmed_values, 6)
    dry_ratio = np.count_nonzero(rounded == 0) / n * 100 if n > 0 else np.nan

    # --- Quartile with most rainfall ---
    if total_acc > 0 and n > 1:
        cum = np.cumsum(trimmed_values) / total_acc
        time_norm = np.linspace(0, 1, n)

        target_pts = np.linspace(0, 1, 5)

        interp_func = interp1d(time_norm,cum,kind='linear',fill_value='extrapolate')

        resampled = interp_func(target_pts)
        fourth_with_most = int(np.argmax(np.diff(resampled)))
    else:
        fourth_with_most = np.nan

    return {'total_acc': total_acc,
            'max_2h': max_acc[2], 'max_3h': max_acc[3], 'max_6h': max_acc[6], 'max_12h': max_acc[12],'max_16h': max_acc[16],
            'peak_position_ratio': peak_position_ratio,'d50': d50,'gini': gini,
            'time_based_std': time_based_std,'dry_ratio_n': dry_ratio,'max_quartile_index': fourth_with_most,
            'length': len(original_values), 
           'length_of_active_rain': len(trimmed_values), 'rainfall_before_peak': rainfall_before_peak, 
           "rainfall_after_peak": rainfall_after_peak, "rainfall_at_peak": rainfall_at_peak,
           "modality": modality_result['modality'], "n_peaks" : modality_result['n_peaks'],
           "peak_share": max_precip/ total_acc, "nonpeak_share" : 1 - (max_precip/ total_acc),
            "loading_index": (rainfall_before_peak -rainfall_after_peak) /total_acc         }

def classify_modality(values, height=10, min_distance=3, wet_threshold=0.05):
    
    if len(values) == 0 or np.max(values) <= 0:
        print("HELP!!")
        return {'n_peaks': 0, 'modality': 'no clear peak', 'peaks': np.array([], dtype=int), 'trimmed_values': values}

    # Short events: explicitly identify the maximum
    if len(values) <= 2:
        peaks = np.array([np.argmax(values)])
    else:
        peaks, _ = find_peaks(values, height=10, distance=min_distance)
        
        # Allow for the peak coming first
        if values[0] >= height and (len(values) == 1 or values[0] > values[1]):
            peaks = np.append(peaks, 0)

        if len(values) > 1 and values[-1] >= height and values[-1] > values[-2]:
            peaks = np.append(peaks, len(values) - 1)

        peaks = np.sort(np.unique(peaks))
        peaks = np.unique(peaks)
        
    n_peaks = len(peaks)
    modality = {0: 'no clear peak', 1: 'unimodal', 2: 'bimodal'}.get(n_peaks, 'multimodal')

    return {'n_peaks': n_peaks, 'modality': modality, 'peaks': peaks, 'trimmed_values': values}


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
from scipy.interpolate import interp1d
from scipy.signal import find_peaks
import os

sys.path.insert(1, '../')
from config import CATCHMENT_LOOKUP_DICT, CATCHMENTS, OUT_DIR # , ENSEMBLE_MEMBERS, CATCHMENTS, OUT_DIR
event_details_fp = 'EventDetails_v4'

In [5]:
print(len(list(CATCHMENT_LOOKUP_DICT.keys())))
for TARGET_HA in ['33_b']: # list(CATCHMENT_LOOKUP_DICT.keys()):
    
    outpath = f"/scratch/hydro4/users/kv25483/FutureFlood/Data/{event_details_fp}/Catchment_{TARGET_HA}/TemporalProfiles.csv"
#     if os.path.exists(outpath):
#         print(TARGET_HA, ": Already exists")
#         continue      

    # Get catchment data
    catchment_name = CATCHMENT_LOOKUP_DICT[TARGET_HA]
    combined_fp_new = os.path.join(OUT_DIR, f"Catchment_{TARGET_HA}", f"{catchment_name}_new_Nov.pkl")
    events_this_catchment = pd.read_pickle(combined_fp_new)
    
    np.set_printoptions(suppress=True)
    metrics = events_this_catchment['rainpeak_rain_series'].apply(compute_temporal_metrics)
    metrics_df = pd.DataFrame(metrics.tolist(), index=events_this_catchment.index)
    
    metrics_df.to_csv(outpath, index=False)

114


In [ ]:
# from sklearn.model_selection import KFold, cross_val_predict
# from sklearn.linear_model import LinearRegression
# from sklearn.metrics import r2_score

# # --- 2. split events by how much rain falls outside the peak step ---
# ev["peak_class"] = pd.cut(ev["peak_share"], [0, 0.5, 0.8, 1.0], labels=["spread", "mixed", "peak-dominated"], include_lowest=True)
# print(ev["peak_class"].value_counts())
 
# # --- 3. does the temporal information help, and only in some classes? (cross-validated R2 with vs without) ---
# base = ["max_precip", "total_acc"]
# temporal = ["rainfall_before_peak", "loading_index"]            # or ["d50_n"]; compare the options
# y = ev["10cm_area"]
 
# def cv_r2(d, cols, target="10cm_area"):
#     p = cross_val_predict(LinearRegression(), d[cols], d[target], cv=KFold(5, shuffle=True, random_state=0))
#     return r2_score(d[target], p)
 
# rows = []
# for cls, g in ev.groupby("peak_class", observed=True):
#     if len(g) < 100: continue
#     rows.append({"class": cls, "n": len(g), "R2_base": cv_r2(g, base), "R2_with_temporal": cv_r2(g, base + temporal),
#                  "R2_with_d50": cv_r2(g, base + ["d50_n"])})
# print(pd.DataFrame(rows).round(3).to_string(index=False))
 
# # --- 4. or one model with an interaction, so the effect scales with how much rain is outside the peak ---
# ev["d50_x_nonpeak"] = ev["d50_n"] * ev["nonpeak_share"]
# print("CV R2, with interaction:", round(cv_r2(ev, base + ["nonpeak_share", "d50_n", "d50_x_nonpeak"]), 3))
 

### Save to file

In [6]:
# fp = f"/scratch/hydro4/users/kv25483/FutureFlood/Data/EventDetails_v4/all_catchments_with_topo_and_temporal.pkl"
# rainfall_events_all_df.to_pickle(fp)